In [2]:
import pandas as pd
import os
from clinlab.funciones import remove_timezones, identify_non_numeric_values, find_impossible_encounters, merge_validating_cardinality, flag_physiological_outliers

ModuleNotFoundError: No module named 'clinlab.funciones'

In [ ]:
PATH_CSV = "/home/jassiel/Documentos/Labs/repo-principal/lab-02/Archivos_synthea"

dtypes_patients = {'Id': 'string', 'GENDER': 'category', 'RACE': 'category'}
dtypes_encounters = {'Id': 'string', 'PATIENT': 'category'}
dtypes_observations = {'PATIENT': 'category', 'ENCOUNTER': 'category', 'VALUE': 'string', 'DESCRIPTION': 'category'}

df_patients = pd.read_csv(os.path.join(PATH_CSV, "patients.csv"), dtype=dtypes_patients, parse_dates=['BIRTHDATE', 'DEATHDATE'], on_bad_lines='skip')
df_encounters = pd.read_csv(os.path.join(PATH_CSV, "encounters.csv"), dtype=dtypes_encounters, parse_dates=['START', 'STOP'], on_bad_lines='skip')
df_observations = pd.read_csv(os.path.join(PATH_CSV, "observations.csv"), dtype=dtypes_observations, parse_dates=['DATE'], on_bad_lines='skip')

In [ ]:
# 1. Quitar zonas horarias
df_encounters = remove_timezones(df_encounters, ['START', 'STOP'])
df_patients = remove_timezones(df_patients, ['BIRTHDATE', 'DEATHDATE'])

# 2. Auditar valores no numéricos
mascara_no_num = identify_non_numeric_values(df_observations['VALUE'])
print(f"Valores no convertibles a número en VALUE: {mascara_no_num.sum()}")

In [ ]:
# 3. Unir validando cardinalidad (m:1)
df_obs_enc = merge_validating_cardinality(df_observations, df_encounters, left_on='ENCOUNTER', right_on='Id', validate_rule='m:1')
df_final = merge_validating_cardinality(df_obs_enc, df_patients, left_on='PATIENT', right_on='Id', validate_rule='m:1')
print(f"Filas tras uniones: {len(df_final)}")

# 4. Detectar fechas imposibles
mascara_fechas_malas = find_impossible_encounters(df_final, start_col='START', birth_col='BIRTHDATE', death_col='DEATHDATE')
print(f"Encuentros imposibles (post-muerte o pre-nacimiento): {mascara_fechas_malas.sum()}")

# 5. Marcar valores implausibles (Hemoglobina fuera de 5-25 g/dL)
mask_hgb = df_final['DESCRIPTION'] == 'Hemoglobin [Mass/volume] in Blood'
outliers_hgb = flag_physiological_outliers(df_final.loc[mask_hgb, 'VALUE'], min_val=5.0, max_val=25.0)
print(f"Valores implausibles de Hemoglobina detectados: {outliers_hgb.sum()}")

In [ ]:
#Pregunta clínica final
df_hemoglobina = df_final[mask_hgb].copy()
df_hemoglobina['VALUE_NUM'] = pd.to_numeric(df_hemoglobina['VALUE'], errors='coerce')

promedio_global = df_hemoglobina['VALUE_NUM'].mean()
promedio_por_paciente = df_hemoglobina.groupby('PATIENT')['VALUE_NUM'].mean()

print(f"Promedio global: {promedio_global:.2f}")
print(f"Promedio de promedios (media poblacional real): {promedio_por_paciente.mean():.2f}")